# Timeseries Analysis Notebook

**NOTE using the de-sar-sample-data envrionment for analysis**
https://github.com/caitlinadams/de-sar-sample-data

In [ ]:
import fsspec
import xarray as xr
import rioxarray
from dotenv import load_dotenv
import re
import pandas as pd
import numpy as np
from scipy.ndimage import uniform_filter
import geopandas as gpd
from dea_tools.plotting import xr_animation
from dea_tools.temporal import xr_optical_flow, xr_regression # !pip install dea-tools==0.4.8dev13
from PIL import Image, ImageSequence
import matplotlib.pyplot as plt
import os
import json
from shapely.geometry import Polygon, mapping
import contextily as ctx
import cv2

In [ ]:
# extra reguirements de-sar-sample-data conda env 
# !pip install dea-tools==0.4.8dev13
# !pip install contextily
# !pip install opencv-python-headless

## Set envrionment credentials for AWS access

In [ ]:
load_dotenv('/home/ec2-user/sar-pipeline/.env')

## Select the burst to assess

In [ ]:
# t007_014550_iw2 (scene 1) - slope and aspect
# t007_014551_iw2 (scene 1) - slope and aspect
# t007_014549_iw1 (scene 1) - elevation
# t007_014543_iw2 (scene 2) - elevation
burst = "t007_014550_iw2"
burst_shape = f"burst_shapefiles/{burst}.json"
burst_shape = gpd.read_file(burst_shape)

## Create a smaller ROI within burst for zoom

In [ ]:
burst_roi = False # if true, assess a zoomed area centered around below. Else assess whole burst.
if burst_roi:
    side_length = 10_000
    burst_roi_centers = {
        't007_014550_iw2' : (-1514900,-640900),
        't007_014551_iw2' : (-1515100,-660600),
        't007_014549_iw1' : (-1482400,-623700),
        't007_014543_iw2' : (-1514772,-508648),
    }

def save_square_geojson(center_x, center_y, side_length, filename):
    """
    Create a square polygon around a centre point (EPSG:3031) and save as GeoJSON.

    Parameters
    ----------
    center_x : float
        X coordinate (EPSG:3031)
    center_y : float
        Y coordinate (EPSG:3031)
    side_length : float
        Length of square's side (same units as EPSG:3031, e.g. metres)
    filename : str
        Output filename for the GeoJSON file

    Returns
    -------
    dict
        A GeoJSON FeatureCollection dictionary
    """
    
    half = side_length / 2.0

    # Define square corners (clockwise or counter-clockwise is fine)
    square = Polygon([
        (center_x - half, center_y - half),
        (center_x + half, center_y - half),
        (center_x + half, center_y + half),
        (center_x - half, center_y + half),
        (center_x - half, center_y - half),
    ])

    feature = {
        "type": "Feature",
        "geometry": mapping(square),
        "properties": {
            "center_x": center_x,
            "center_y": center_y,
            "side_length": side_length,
            "crs": "EPSG:3031"
        }
    }

    feature_collection = {
        "type": "FeatureCollection",
        "features": [feature]
    }

    # Save to file
    with open(filename, "w") as f:
        json.dump(feature_collection, f, indent=2)

    return feature_collection

if burst_roi:
    roi_x, roi_y = burst_roi_centers[burst]
    burst_roi_shape = f'burst_shapefiles/{burst}_{roi_x}_{roi_y}.json'
    save_square_geojson(roi_x, roi_y, side_length, burst_roi_shape)
    burst_roi_shape = gpd.read_file(burst_roi_shape)
    burst_roi_shape = burst_roi_shape.set_crs(epsg=3031, inplace=False,allow_override=True)
    
    # plot to see overlap
    fig, ax = plt.subplots(figsize=(8,8))
    burst_shape_3031 = burst_shape.to_crs(epsg=3031)
    burst_roi_shape.plot(ax=ax, color='red', alpha=0.5, edgecolor='black', label="ROI")
    burst_shape_3031.plot(ax=ax, color='blue', alpha=0.5, edgecolor='black', label=f"{burst}")
    ax.legend()
    ax.set_title(f"Burst : {burst} and ROI overlap")
    plt.show()
    

## Functions

In [ ]:
# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    filtered_data_masked = xr.where(np.isnan(data_array), np.nan, filtered_data)

    return filtered_data_masked


def slope_aspect(dem, dx=1.0, dy=1.0):
    """
    Compute slope and aspect from a 2D DEM array, handling nodata values.
    
    Parameters
    ----------
    dem : np.ndarray
        2D elevation array (NaNs represent nodata).
    dx : float
        Spatial resolution in x-direction.
    dy : float
        Spatial resolution in y-direction.
    
    Returns
    -------
    slope : np.ndarray
        Slope in degrees.
    aspect : np.ndarray
        Aspect in degrees (0 = North, clockwise).
    """
    # Mask invalid values
    dem = np.where(np.isfinite(dem), dem, np.nan)
    
    # Compute gradients
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    
    # Replace NaN gradients with 0 to avoid NaNs in slope/aspect
    dz_dx = np.nan_to_num(dz_dx)
    dz_dy = np.nan_to_num(dz_dy)
    
    # Slope in degrees
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)) * 180 / np.pi
    
    # Aspect in degrees
    aspect = np.arctan2(-dz_dx, -dz_dy) * 180 / np.pi
    aspect = np.where(np.isnan(dem), np.nan, (aspect + 360) % 360)
    
    return slope, aspect


def slope_aspect_timeseries(ds_dem, dx=1.0, dy=1.0):
    """
    Apply slope_aspect function to an xarray DataArray over time,
    preserving NaNs for nodata values.
    
    Parameters
    ----------
    ds_dem : xarray.DataArray
        DEM time series with dimensions ('time', 'y', 'x')
    dx, dy : float
        Spatial resolution
    
    Returns
    -------
    slopes : xarray.DataArray
        Slope for each timestep
    aspects : xarray.DataArray
        Aspect for each timestep
    """
    slopes, aspects = xr.apply_ufunc(
        slope_aspect,
        ds_dem,
        dx,
        dy,
        input_core_dims=[["y", "x"], [], []],
        output_core_dims=[["y", "x"], ["y", "x"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[ds_dem.dtype, ds_dem.dtype],
    )
    
    slopes = xr.DataArray(slopes, coords=ds_dem.coords, dims=ds_dem.dims, name="slope")
    aspects = xr.DataArray(aspects, coords=ds_dem.coords, dims=ds_dem.dims, name="aspect")
    
    return slopes, aspects

def keep_s3_prefix(file_list):
    return [f if f.startswith("s3://") else f"s3://{f}" for f in file_list]

# Extract timestamps from filenames (e.g., 20170405T050842)
def extract_time(fp):
    match = re.search(r"(\d{8}T\d{6})", fp)
    return pd.to_datetime(match.group(1)) if match else None

def load_xr_dataset(file_list):
    datasets = []
    for fp in file_list:
        da = rioxarray.open_rasterio(
            fp,
            masked=True,
            chunks=True,
        )
        da = da.squeeze().expand_dims(time=[extract_time(fp)])  # add time dim
        datasets.append(da)

        # Combine along the time dimension
    return xr.concat(datasets, dim="time")  # dims: ('time', 'y', 'x')

def preprocess_nrb_xr_dataset(ds_nrb, burst_shape, cal = 'gamma0'):
    # Open each file lazily with rioxarray and assign a time coordinate
    ds_nrb.name = f"HH_{cal}"
    # trim the dataset withg burst shapefule
    burst_shape = burst_shape.to_crs(ds_nrb.rio.crs)
    ds_nrb = ds_nrb.rio.clip(burst_shape.geometry, crs=ds_nrb.rio.crs, drop=True)
    ds_nrb.odc.assign_crs(crs='EPSG:3031')
    # Apply Lee filter directly on the DataArray
    ds_nrb[f"HH_{cal}_filtered"] = apply_lee_filter(ds_nrb, size=5)
    # Convert to dB only for positive values
    ds_nrb[f"HH_{cal}_db"] = xr.where(
        ds_nrb.to_dataset(name=f'HH_{cal}')[f'HH_{cal}'] > 0,
        10 * np.log10(ds_nrb.to_dataset(name=f'HH_{cal}')[f'HH_{cal}']),
        np.nan
    )
    ds_nrb[f"HH_{cal}_filtered_db"] = 10 * np.log10(ds_nrb[f'HH_{cal}_filtered'])
    return ds_nrb

def preprocess_dem_xr_dataset(ds_dem, burst_shape):
    ds_dem.name = "dem"
    burst_shape = burst_shape.to_crs(ds_dem.rio.crs)
    ds_dem = ds_dem.rio.clip(burst_shape.geometry, crs=ds_dem.rio.crs, drop=True)
    ds_dem.odc.assign_crs(crs='EPSG:3031')
    ds_dem = ds_dem.to_dataset(name="elevation")
    slopes, aspects = slope_aspect_timeseries(ds_dem['elevation'], dx=20, dy=20)
    ds_dem = ds_dem.assign({
        "slope": slopes,
        "aspect": aspects
    })
    return ds_dem


## Load in the xarrays

In [ ]:
if not burst_roi:
    results_folder = f'timeseries-results/gamma0_vs_beta0/{burst}'
else:
    results_folder = f'timeseries-results/gamma0_vs_beta0/{burst}_{roi_x}_{roi_y}'
os.makedirs(results_folder, exist_ok=True)

### Load in Both Gamma0 and Beta0 for same timeseries

In [ ]:
# Create S3 filesystem (anonymous access)
fs = fsspec.filesystem("s3", anon=True)

# Base S3 folder (public)
gamma0_timeseries_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES/ga_s1_nrb_iw_hh_0/{burst}/"
beta0_timeseries_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES_BETA0/ga_s1_nrb_iw_hh_0/{burst}/"

# Find all GeoTIFFs recursively
gamma0_timeseries_nrb_tif_files = fs.glob(f"{gamma0_timeseries_base_path}**/*HH-gamma0.tif")
gamma0_timeseries_dem_tif_files = fs.glob(f"{gamma0_timeseries_base_path}**/*digital-elevation-model.tif")
beta0_timeseries_nrb_tif_files = fs.glob(f"{beta0_timeseries_base_path}**/*HH-beta0.tif")
beta0_timeseries_dem_tif_files = fs.glob(f"{beta0_timeseries_base_path}**/*digital-elevation-model.tif")

# Ensure each file keeps the s3:// prefix (sometimes fsspec strips it)
gamma0_timeseries_nrb_tif_files = keep_s3_prefix(gamma0_timeseries_nrb_tif_files)
gamma0_timeseries_dem_tif_files = keep_s3_prefix(gamma0_timeseries_dem_tif_files)
beta0_timeseries_nrb_tif_files = keep_s3_prefix(beta0_timeseries_nrb_tif_files)
beta0_timeseries_dem_tif_files = keep_s3_prefix(beta0_timeseries_dem_tif_files)

print(f"Found {len(gamma0_timeseries_nrb_tif_files)} Gamma0 timeseries NRB TIFs:")
print(f"Found {len(gamma0_timeseries_dem_tif_files)} Gamma0 timeseries DEM TIFs:")
print(f"Found {len(beta0_timeseries_nrb_tif_files)} Beta0 timeseries NRB TIFs:")
print(f"Found {len(beta0_timeseries_dem_tif_files)} Beta0 timeseries DEM TIFs:")

# get the shape for trimming the xarray
clip_shape = burst_roi_shape if burst_roi else burst_shape

# load and preprocess the nrb datasets
ds_gamma0_timeseries_nrb = load_xr_dataset(gamma0_timeseries_nrb_tif_files)
ds_gamma0_timeseries_nrb = preprocess_nrb_xr_dataset(ds_gamma0_timeseries_nrb, clip_shape, cal="gamma0")
ds_beta0_timeseries_nrb = load_xr_dataset(beta0_timeseries_nrb_tif_files)
ds_beta0_timeseries_nrb = preprocess_nrb_xr_dataset(ds_beta0_timeseries_nrb, clip_shape, cal="beta0")

# load in the dem datasets
ds_gamma0_timeseries_dem = load_xr_dataset(gamma0_timeseries_dem_tif_files)
ds_gamma0_timeseries_dem = preprocess_dem_xr_dataset(ds_gamma0_timeseries_dem, clip_shape)
ds_beta0_timeseries_dem = load_xr_dataset(beta0_timeseries_dem_tif_files)
ds_beta0_timeseries_dem = preprocess_dem_xr_dataset(ds_beta0_timeseries_dem, clip_shape)

In [ ]:
ds_gamma0_timeseries_nrb.isel(time=1)[f'HH_gamma0_filtered_db'].plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(12,5)) 
ds_beta0_timeseries_nrb.isel(time=1)[f'HH_beta0_filtered_db'].plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(12,5)) 

In [ ]:
make_animation = True
if make_animation:
    for i,ds in enumerate([ds_gamma0_timeseries_nrb,ds_gamma0_timeseries_dem,ds_beta0_timeseries_nrb,ds_beta0_timeseries_dem]):
        dem_type = ['REMA_10_TIMESERIES','REMA_10_TIMESERIES','REMA_10_TIMESERIES','REMA_10_TIMESERIES'][i]
        plot_var = [f'HH_gamma0_filtered_db','elevation',f'HH_beta0_filtered_db','elevation'][i]
        titles = [f'gamma0 (dB)', 'Elevation (m)', f'beta0 (dB)', 'Elevation (m)']
        if plot_var in [f'HH_gamma0_filtered_db','HH_beta0_filtered_db']:
            imshow_kwargs={"cmap":"bone", "vmin":-20, "vmax":0}
            plot_ds = ds[plot_var].to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
            band = ds.name
        if plot_var == 'elevation':
            imshow_kwargs={}
            plot_ds = ds[plot_var].to_dataset(name='elevation').odc.assign_crs(crs='EPSG:3031')
            band = plot_var
        xr_animation(
            plot_ds,
            bands=band,
            output_path=f'{results_folder}/{burst}_{plot_var}_{dem_type}.gif',
            width_pixels=1200,
            interval=300,
            show_date='%d %b %Y',
            show_text=f'{titles[i]} - burst: {burst}  dem_type: {dem_type}',
            show_colorbar=True,
            imshow_kwargs=imshow_kwargs,
            colorbar_kwargs={'colors': 'black'},
        )

## Animation of NRB differnces at each timestep

In [ ]:
# Compute difference at each timestep
nrb_diff_ts = ds_gamma0_timeseries_nrb[f'HH_gamma0_filtered_db'] - ds_beta0_timeseries_nrb[f'HH_beta0_filtered_db']
nrb_diff_ds = xr.Dataset({f"HH_difference_filtered_db_diff": nrb_diff_ts})

In [ ]:
make_animation = True
#.to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
if make_animation:
    xr_animation(
        nrb_diff_ds.odc.assign_crs(crs='EPSG:3031'),
        bands=f"HH_difference_filtered_db_diff",
        output_path=f'{results_folder}/{burst}_gamma0_minus_beta0_difference.gif',
        width_pixels=1200,
        interval=300,
        show_date='%d %b %Y',
        show_text=f' burst: {burst}\nGamma0 minus Beta0 Difference (dB)\n(difference introduced by radiometric correction)',
        show_colorbar=True,
        imshow_kwargs={"cmap":"RdBu", "vmin":-5, "vmax":5},
        colorbar_kwargs={'colors': 'black'},
    )

## Animation of NRB differnces at each timestep

In [ ]:
# Compute difference at each timestep
dem_diff_ts = ds_gamma0_timeseries_dem.elevation - ds_beta0_timeseries_dem.elevation
dem_diff_ds = xr.Dataset({"elevation_m_diff": dem_diff_ts})

In [ ]:
make_animation = True
#.to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
if make_animation:
    xr_animation(
        dem_diff_ds.odc.assign_crs(crs='EPSG:3031'),
        bands="elevation_m_diff",
        output_path=f'{results_folder}/{burst}_gamma0_vs_beta0_dem_difference.gif',
        width_pixels=1200,
        interval=300,
        show_date='%d %b %Y',
        show_text=f'Elevation Difference (m) - burst: {burst}',
        show_colorbar=True,
        imshow_kwargs={"cmap":"RdBu", "vmin":-1, "vmax":1},
        colorbar_kwargs={'colors': 'black'},
    )